Classificatore Binario

MLP con BCEWithLogitsLoss

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class ClassificatoreBinario(nn.Module):
    """
    MLP per classificazione binaria.
    Uscita: logit scalare (NO sigmoid -- assorbita da BCEWithLogitsLoss).
    """
    def __init__(self, input_dim: int, hidden_dim: int = 64, dropout_p: float = 0.5):
        super().__init__()
        self.rete = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim // 2, 1) # logit scalare
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.rete(x).squeeze(-1) # shape: (batch,)

def addestra_classificatore_binario(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    input_dim: int,
    n_epochs: int = 50,
    lr: float = 1e-3,
    batch_size: int = 32,
    pos_weight: torch.Tensor = None
) -> ClassificatoreBinario:
    """
    Addestramento con BCEWithLogitsLoss (numericamente stabile).
    pos_weight: tensore scalare per gestire lo sbilanciamento delle classi.
    """
    dataset = TensorDataset(X_train, y_train.float())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    modello = ClassificatoreBinario(input_dim)
    criterio = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    ottimizza = torch.optim.Adam(modello.parameters(), lr=lr)

    modello.train()
    for epoch in range(n_epochs):
        perdita_media = 0.0
        for X_batch, y_batch in loader:
            ottimizza.zero_grad()
            logit = modello(X_batch)
            perdita = criterio(logit, y_batch)
            perdita.backward()
            ottimizza.step()
            perdita_media += perdita.item()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{n_epochs} -- "
                  f"Loss: {perdita_media / len(loader):.4f}")

    return modello

def predici_binario(
    modello: ClassificatoreBinario,
    X_test: torch.Tensor,
    soglia: float = 0.5
):
    """Converte i logit in probabilità e applica la soglia decisionale."""
    modello.eval()
    with torch.no_grad():
        logit = modello(X_test)
        prob = torch.sigmoid(logit) # probabilità in (0, 1)
        pred = (prob >= soglia).long()

    return pred, prob

Classificatore multiclasse in PyTorch

Con  CrossEntropyLoss

In [ ]:
import torch
import torch.nn as nn

class ClassificatoreMulticlasse(nn.Module):
    """
    MLP per classificazione in K classi.
    Uscita: K logit grezzi (softmax incorporata in CrossEntropyLoss).
    """
    def __init__(self, input_dim: int, n_classi: int, hidden_dims: list = [128, 64]):
        super().__init__()
        strati = []
        dim_in = input_dim

        for dim_h in hidden_dims:
            strati += [
                nn.Linear(dim_in, dim_h),
                nn.BatchNorm1d(dim_h),
                nn.ReLU(),
                nn.Dropout(0.5)
            ]
            dim_in = dim_h

        strati.append(nn.Linear(dim_in, n_classi)) # K logit
        self.rete = nn.Sequential(*strati)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.rete(x) # shape: (batch, K) -- logit grezzi

# =====================================================================
# Esempio di Loss e funzione di probabilità
# =====================================================================

# CrossEntropyLoss = LogSoftmax + NLLLoss (numericamente stabile).
# ATTENZIONE: i logit non devono essere passati a softmax prima della loss.
# Target: indici interi delle classi in {0, ... , K-1} (torch.long).

# Inizializzazione per un problema a 5 classi con sbilanciamento:
pesi_classi = torch.tensor([1.0, 2.0, 1.5, 3.0, 1.0])
criterio = nn.CrossEntropyLoss(
    weight=pesi_classi,     # peso per classe sbilanciata
    label_smoothing=0.1     # regolarizzazione con label smoothing
)

# Probabilità di uscita per inferenza:
def probabilita_classi(modello: nn.Module, X: torch.Tensor):
    modello.eval()
    with torch.no_grad():
        logit = modello(X)
        return torch.softmax(logit, dim=-1) # shape: (batch, K)

Regressione Neurale

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RegressoreNeurale(nn.Module):
    """
    MLP per regressione a singolo output.
    Uscita: scalare reale (strato lineare finale, nessuna attivazione).
    """
    def __init__(self, input_dim: int, hidden_dims: list = [128, 64]):
        super().__init__()
        strati = []
        dim_in = input_dim

        for dim_h in hidden_dims:
            strati += [
                nn.Linear(dim_in, dim_h),
                nn.LayerNorm(dim_h),
                nn.GELU()
            ]
            dim_in = dim_h

        strati.append(nn.Linear(dim_in, 1)) # uscita lineare
        self.rete = nn.Sequential(*strati)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.rete(x).squeeze(-1) # shape: (batch,)


class RegressoreConIncertezza(nn.Module):
    """
    MLP con uscita doppia: media hat_mu e deviazione std hat_sigma.
    Perdita: Negative Log-Likelihood gaussiana.
    """
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU()
        )
        self.testa_mu = nn.Linear(hidden_dim // 2, 1)
        self.testa_sigma = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x: torch.Tensor):
        h = self.backbone(x)
        mu = self.testa_mu(h).squeeze(-1)

        # softplus garantisce sigma > 0 (aggiunta epsilon per stabilità numerica)
        sigma = F.softplus(self.testa_sigma(h)).squeeze(-1) + 1e-6
        return mu, sigma

def perdita_nll_gaussiana(
    mu: torch.Tensor,
    sigma: torch.Tensor,
    y: torch.Tensor
) -> torch.Tensor:
    """ Negative Log-Likelihood gaussiana (eq. della dispensa). """
    return (0.5 * ((y - mu) / sigma)**2 + torch.log(sigma)).mean()

# --- Confronto MSE vs Huber ---

if __name__ == "__main__":
    X = torch.randn(200, 10)
    y = torch.randn(200)

    mod = RegressoreNeurale(input_dim=10)
    pred = mod(X)

    mse_loss = nn.MSELoss()
    huber_loss = nn.SmoothL1Loss(beta=1.0) # beta corrisponde a delta

    print(f"MSE Loss: {mse_loss(pred, y).item():.4f}")
    print(f"Huber Loss: {huber_loss(pred, y).item():.4f}")

Pipeline completa con scikit-learn

Scikit-learn offre implementazioni pronte all'uso tramite MLPClassifier e MLPRegressor

In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             mean_squared_error, r2_score)
from sklearn.pipeline import Pipeline

# -------------------------------------------------------
# 1. Classificazione binaria con MLPClassifier
# -------------------------------------------------------
np.random.seed(42)
X = np.random.randn(500, 20)
y_cls = (X[:, 0] + X[:, 1] > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_cls, test_size=0.2, stratify=y_cls, random_state=42
)

# Pipeline: StandardScaler + MLPClassifier
# StandardScaler è quasi sempre necessario per le reti neurali
pipeline_cls = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        max_iter=500,
        early_stopping=True, # split interno di validazione
        validation_fraction=0.1,
        random_state=42
    ))
])

pipeline_cls.fit(X_tr, y_tr)
y_pred = pipeline_cls.predict(X_te)
y_prob = pipeline_cls.predict_proba(X_te)[:, 1]

print("=== Classificazione Binaria ===")
print(classification_report(y_te, y_pred, target_names=['Classe 0', 'Classe 1']))
print(f"AUC-ROC: {roc_auc_score(y_te, y_prob):.4f}")

# -------------------------------------------------------
# 2. Regressione con MLPRegressor
# -------------------------------------------------------
y_reg = 2.5 * X[:, 0] - 1.2 * X[:, 2] + np.random.randn(500) * 0.1

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

pipeline_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        max_iter=500,
        early_stopping=True,
        random_state=42
    ))
])

pipeline_reg.fit(X_tr_r, y_tr_r)
y_pred_r = pipeline_reg.predict(X_te_r)

print("\n=== Regressione ===")
print(f"MSE: {mean_squared_error(y_te_r, y_pred_r):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_te_r, y_pred_r)):.4f}")
print(f"R2: {r2_score(y_te_r, y_pred_r):.4f}")